# HandwrittenOCR - استخراج وتصحيح نصوص الخط اليدوي

دفتر Google Colab للعمل مع المشروع.

---

## الخلية 1: تثبيت المكتبات المطلوبة

In [ ]:
!apt-get install -y poppler-utils tesseract-ocr
!pip install -q pdf2image easyocr pyspellchecker langdetect pytesseract \
    transformers torch torchvision Pillow opencv-python-headless \
    pandas ar-corrector ipywidgets

## الخلية 2: الاستيرادات وإعداد المسارات

In [ ]:
import os

# === جلب HF_TOKEN من Colab Secrets ===
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    os.environ["HF_TOKEN"] = HF_TOKEN
    print("تم تحميل Hugging Face Token من Colab Secrets")
except (ImportError, userdata.SecretNotFoundError):
    HF_TOKEN = ""
    print("لم يتم العثور على HF_TOKEN في Secrets - سيتم استخدام الوصول العام")
except Exception as e:
    HF_TOKEN = ""
    print(f"خطأ في جلب التوكن: {e}")

# === ربط Google Drive ===
from google.colab import drive
drive.mount('/content/drive')

# === إعداد المسارات ===
BASE_DIR = '/content/drive/MyDrive'
PDF_PATH = os.path.join(BASE_DIR, 'python notes.pdf')
OUTPUT_DIR = os.path.join(BASE_DIR, 'Handwriting_Dataset')
LOGS_DIR = os.path.join(OUTPUT_DIR, 'Logs')
MODEL_CACHE = os.path.join(OUTPUT_DIR, 'models_cache')

os.makedirs(LOGS_DIR, exist_ok=True)
os.makedirs(MODEL_CACHE, exist_ok=True)

# تطبيق مسارات التخزين المؤقت
os.environ["TRANSFORMERS_CACHE"] = MODEL_CACHE
os.environ["TORCH_HOME"] = MODEL_CACHE

# === إعداد قاعدة البيانات والملفات ===
DB_PATH = os.path.join(OUTPUT_DIR, 'handwriting_data.db')
LOG_FILE = os.path.join(LOGS_DIR, f"ocr_log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log")
FEEDBACK_CSV = os.path.join(LOGS_DIR, "user_corrections_feedback.csv")
STATS_JSON = os.path.join(LOGS_DIR, "processing_stats.json")

# === إعداد logging ===
import cv2, numpy as np, sqlite3, io, torch, json, time, logging
import pandas as pd
from PIL import Image
from pdf2image import convert_from_path
from google.colab import patches
from datetime import datetime

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[logging.FileHandler(LOG_FILE, encoding='utf-8'), logging.StreamHandler()]
)

if not os.path.exists(FEEDBACK_CSV):
    pd.DataFrame(
        columns=['timestamp', 'image_id', 'original_text', 'corrected_text', 'status']
    ).to_csv(FEEDBACK_CSV, index=False, encoding='utf-8')

logging.info("بدء تشغيل المشروع")
logging.info(f"سجل الأحداث: {LOG_FILE}")
logging.info(f"ملف التصحيحات: {FEEDBACK_CSV}")

## الخلية 3: تحميل النماذج (TrOCR، EasyOCR، المدققات)

In [ ]:
import easyocr
import os.path as osp
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from spellchecker import SpellChecker
from langdetect import detect, DetectorFactory
from ar_corrector.corrector import Corrector

print("جاري فحص النماذج المحملة مسبقاً...")
start_time = time.time()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logging.info(f"الجهاز المستخدم: {device}")

# === TrOCR مع cache_dir ===
try:
    trocr_processor = TrOCRProcessor.from_pretrained(
        "David-Magdy/TR_OCR_LARGE", cache_dir=MODEL_CACHE
    )
    trocr_model = VisionEncoderDecoderModel.from_pretrained(
        "David-Magdy/TR_OCR_LARGE", cache_dir=MODEL_CACHE
    ).to(device)
    print("TrOCR جاهز")
except Exception as e:
    print(f"فشل تحميل TrOCR: {e}")

# === EasyOCR ===
EASYOCR_DIR = osp.expanduser("~/.EasyOCR")
detect_model_path = osp.join(EASYOCR_DIR, "model/craft_mlt_25k.pth")
recog_model_path = osp.join(EASYOCR_DIR, "model/english_g2.pth")

if osp.exists(detect_model_path) and osp.exists(recog_model_path):
    print("EasyOCR: نماذج موجودة مسبقاً")
else:
    print("سيتم تنزيل نماذج EasyOCR (مرة واحدة)")

easy_reader = easyocr.Reader(['en', 'ar'])

# === نقل نماذج EasyOCR إلى Drive وإنشاء symlink ===
drive_easyocr_path = os.path.join(OUTPUT_DIR, '.EasyOCR')
local_easyocr_path = os.path.expanduser('~/.EasyOCR')

if not os.path.exists(drive_easyocr_path) and os.path.exists(local_easyocr_path):
    import shutil
    print('جاري نقل نماذج EasyOCR إلى Drive للمرة الأولى...')
    shutil.move(local_easyocr_path, drive_easyocr_path)

if not os.path.islink(local_easyocr_path):
    if os.path.exists(local_easyocr_path):
        shutil.rmtree(local_easyocr_path)
    os.symlink(drive_easyocr_path, local_easyocr_path)
    print('تم ربط نماذج EasyOCR بـ Google Drive بنجاح')

# === المدققات الإملائية ===
arabic_spell = Corrector()
english_spell = SpellChecker(language='en')
DetectorFactory.seed = 0

logging.info(f"تم التحميل في {time.time()-start_time:.2f} ثانية")
print(f"تم تحميل جميع النماذج في {time.time()-start_time:.2f} ثانية")

## الخلية 4: دوال المعالجة المسبقة والتعرف والتصحيح

**تصحيحات مطبقة:**
- `preprocess_image` ترجع `(binary, enhanced)` بدلاً من binary فقط
- `recognize_word` تستقبل الصورة BGR الأصلية (لا الثنائية)
- `correct_text` تصحح الجمل الإنجليزية كلمة بكلمة مع حفظ الترقيم
- `process_pdf` يقطع الكلمات من الصورة الأصلية BGR

In [ ]:
def preprocess_image(img_bgr):
    """
    تحسين الصورة: تسوية الميل، CLAHE، إزالة الضوضاء، thresholding.
    تصحيح: ترجع (binary, enhanced) بدلاً من binary فقط.
    """
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

    # Deskewing
    coords = np.column_stack(np.where(gray > 0))
    if len(coords) > 100:
        angle = cv2.minAreaRect(coords)[-1]
        if angle < -45:
            angle += 90
        if abs(angle) > 0.5:
            h, w = gray.shape
            M = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1.0)
            gray = cv2.warpAffine(gray, M, (w, h), flags=cv2.INTER_CUBIC,
                                  borderMode=cv2.BORDER_REPLICATE)

    # CLAHE
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(gray)

    # إزالة الضوضاء + Thresholding
    denoised = cv2.fastNlMeansDenoising(enhanced, h=30)
    _, binary = cv2.threshold(denoised, 0, 255,
                              cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    return binary, enhanced


def recognize_word(img_bgr):
    """
    التعرف باستخدام TrOCR أولاً، ثم EasyOCR كبديل.
    تصحيح: تستقبل الصورة BGR الأصلية (ليس الثنائية).
    تصحيح: EasyOCR يختار النص بأعلى ثقة.
    """
    # TrOCR
    try:
        rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        pix = trocr_processor(images=Image.fromarray(rgb),
                              return_tensors="pt").pixel_values.to(device)
        ids = trocr_model.generate(pix, max_length=50)
        txt = trocr_processor.batch_decode(ids, skip_special_tokens=True)[0].strip()
        if len(txt) > 1:
            return txt
    except Exception as e:
        logging.debug(f"TrOCR فشل: {e}")

    # EasyOCR
    try:
        res = easy_reader.readtext(img_bgr)
        if res:
            best = max(res, key=lambda r: r[2])  # تصحيح: أعلى ثقة
            return best[1]
    except Exception as e:
        logging.debug(f"EasyOCR فشل: {e}")

    return ""


def correct_text(text):
    """
    تصحيح إملائي حسب اللغة.
    تصحيح: الجمل الإنجليزية تُصحح كلمة بكلمة مع حفظ الترقيم.
    """
    if not text:
        return ""
    try:
        lang = detect(text)
        if lang == 'ar':
            return arabic_spell.contextual_correct(text)
        elif lang == 'en':
            words = text.split()
            corrected = []
            for word in words:
                clean = word.strip('.,;:!?\"\'()-')
                if clean:
                    fixed = english_spell.correction(clean)
                    corrected_word = word.replace(clean, fixed) if fixed else word
                    corrected.append(corrected_word)
                else:
                    corrected.append(word)
            return " ".join(corrected)
    except Exception:
        pass
    return text

## الخلية 5: دالة معالجة PDF وحفظ البيانات في SQLite

**تصحيح:** الكلمات تُقطع من الصورة الأصلية `img_bgr` (ليس من الصورة الثنائية)

In [ ]:
def process_pdf(pages_start=1, pages_end=2):
    """
    استخراج الكلمات من PDF، حفظها في قاعدة البيانات مع إحصائيات.
    تصحيح: الكلمات تُقطع من img_bgr الأصلية (لا من الصورة الثنائية).
    """
    proc_start = time.time()
    logging.info(f"بدء معالجة الصفحات {pages_start} إلى {pages_end}")

    try:
        images = convert_from_path(PDF_PATH, dpi=300,
                                   first_page=pages_start, last_page=pages_end)
        logging.info(f"تم تحويل {len(images)} صفحة")
    except Exception as e:
        logging.error(f"فشل تحويل PDF: {e}")
        return

    with sqlite3.connect(DB_PATH) as conn:
        conn.execute('''CREATE TABLE IF NOT EXISTS handwriting_data
                        (image_id INTEGER PRIMARY KEY AUTOINCREMENT,
                         image_data BLOB, predicted_text TEXT, status TEXT)''')
        total_words = 0
        failed_ocr = 0
        corrected_by_spell = 0

        for idx, pil in enumerate(images):
            logging.info(f"معالجة صفحة {idx+1}/{len(images)}")
            img_bgr = cv2.cvtColor(np.array(pil), cv2.COLOR_RGB2BGR)

            # تصحيح: preprocess_image ترجع (binary, enhanced)
            binary, enhanced = preprocess_image(img_bgr)

            # تجزئة الكلمات باستخدام الكنتورات
            kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (25, 5))
            dilated = cv2.dilate(binary, kernel, iterations=1)
            contours, _ = cv2.findContours(dilated, cv2.RETR_EXTERNAL,
                                           cv2.CHAIN_APPROX_SIMPLE)

            preview = img_bgr.copy()
            for cnt in contours:
                x, y, w, h = cv2.boundingRect(cnt)
                if w < 25 or h < 15:
                    continue

                # تصحيح: قص من الصورة الأصلية BGR (ليس من الثنائية)
                crop = img_bgr[y:y+h, x:x+w]
                raw = recognize_word(crop)
                if not raw:
                    failed_ocr += 1
                final = correct_text(raw)
                if raw and raw != final:
                    corrected_by_spell += 1

                _, buf = cv2.imencode(".png", crop)
                conn.execute(
                    'INSERT INTO handwriting_data '
                    '(image_data, predicted_text, status) VALUES (?,?,?)',
                    (buf.tobytes(), final, 'yes')
                )
                cv2.rectangle(preview, (x, y), (x+w, y+h), (0, 255, 0), 2)
                total_words += 1

            patches.cv2_imshow(cv2.resize(preview, (0, 0), fx=0.4, fy=0.4))
        conn.commit()

    elapsed = time.time() - proc_start
    stats = {
        "timestamp": datetime.now().isoformat(),
        "pages_processed": len(images),
        "total_words": total_words,
        "ocr_failures": failed_ocr,
        "spell_corrections": corrected_by_spell,
        "time_seconds": elapsed,
        "avg_time_per_word": elapsed / total_words if total_words else 0
    }
    with open(STATS_JSON, "w", encoding='utf-8') as f:
        json.dump(stats, f, indent=4, ensure_ascii=False)

    logging.info(f"اكتملت المعالجة: {total_words} كلمة في {elapsed:.2f} ثانية")

## الخلية 6: تشغيل عملية الاستخراج

In [ ]:
# اختر نطاق الصفحات (مثال: من 1 إلى 2)
process_pdf(pages_start=1, pages_end=2)

## الخلية 7: الواجهة التفاعلية المتكاملة (مع الحذف، التصحيح، وتسجيل feedback)

In [ ]:
import ipywidgets as widgets
from IPython.display import display

def launch_review_ui():
    with sqlite3.connect(DB_PATH) as conn:
        df = pd.read_sql_query('SELECT * FROM handwriting_data', conn)

    if df.empty:
        print("لا توجد بيانات. تأكد من تشغيل خلية المعالجة أولاً.")
        return

    current_index = 0

    img_w = widgets.Image(format='png', width=350)
    txt_i = widgets.Text(description="النص الصحيح:", layout=widgets.Layout(width='95%'))
    stat_c = widgets.Checkbox(value=True, description="تضمين في التدريب")
    prog = widgets.IntProgress(min=0, max=len(df)-1, bar_style='info',
                               layout=widgets.Layout(width='95%'))
    info = widgets.Label()

    def log_correction(img_id, original, corrected, status):
        pd.DataFrame([{
            'timestamp': datetime.now().isoformat(),
            'image_id': img_id,
            'original_text': original,
            'corrected_text': corrected,
            'status': status
        }]).to_csv(FEEDBACK_CSV, mode='a', header=False, index=False, encoding='utf-8')
        logging.info(f"Feedback: ID={img_id}, '{original}' -> '{corrected}'")

    def update_view():
        nonlocal current_index
        if 0 <= current_index < len(df):
            row = df.iloc[current_index]
            img_w.value = row['image_data']
            txt_i.value = row['predicted_text'] or ""
            stat_c.value = (row['status'] == 'yes')
            prog.value = current_index
            info.value = f"السجل {current_index+1} من {len(df)} (ID: {row['image_id']})"

    def on_next(b):
        nonlocal current_index
        rid = int(df.iloc[current_index]['image_id'])
        original = df.iloc[current_index]['predicted_text'] or ""
        corrected = txt_i.value
        new_status = 'yes' if stat_c.value else 'no'

        with sqlite3.connect(DB_PATH) as conn:
            conn.execute(
                'UPDATE handwriting_data SET predicted_text=?, status=? WHERE image_id=?',
                (corrected, new_status, rid)
            )
        if original != corrected:
            log_correction(rid, original, corrected, new_status)

        current_index = min(len(df)-1, current_index + 1)
        update_view()

    def on_prev(b):
        nonlocal current_index
        current_index = max(0, current_index - 1)
        update_view()

    def on_delete(b):
        nonlocal current_index, df
        if current_index < len(df):
            rid = int(df.iloc[current_index]['image_id'])
            with sqlite3.connect(DB_PATH) as conn:
                conn.execute('DELETE FROM handwriting_data WHERE image_id=?', (rid,))
            df = df.drop(current_index).reset_index(drop=True)
            prog.max = max(0, len(df)-1)
            if current_index >= len(df) and current_index > 0:
                current_index = len(df) - 1
            update_view()

    btn_next = widgets.Button(description="التالي", button_style='success')
    btn_prev = widgets.Button(description="السابق", button_style='info')
    btn_del = widgets.Button(description="حذف", button_style='danger')
    btn_next.on_click(on_next)
    btn_prev.on_click(on_prev)
    btn_del.on_click(on_delete)

    ui = widgets.VBox([
        widgets.HTML("<h3>مراجعة وتصحيح نصوص الخط اليدوي</h3>"),
        prog, info,
        widgets.Box([img_w], layout=widgets.Layout(
            display='flex', justify_content='center', padding='10px')),
        txt_i, stat_c,
        widgets.HBox([btn_prev, btn_del, btn_next])
    ])
    display(ui)
    update_view()

launch_review_ui()

## الخلية 8: عرض مسارات ملفات المراقبة

In [ ]:
print("\nملفات المراقبة والتطوير:")
print(f"   سجل الأحداث: {LOG_FILE}")
print(f"   إحصائيات المعالجة: {STATS_JSON}")
print(f"   تصحيحات المستخدم: {FEEDBACK_CSV}")
print(f"   قاعدة البيانات: {DB_PATH}")
print(f"   تخزين النماذج: {MODEL_CACHE}")